# 001 — Entropy-Gated Local Learning

**Core idea:** Each neuron controls its own plasticity based on its local entropy.  
**Result:** 97.46% on MNIST without backpropagation (backprop = 98.04%).

---

## Table of Contents
1. Setup & Imports
2. Algorithm Explanation
3. Training Run (MNIST)
4. Analysis: Entropy & Plasticity Dynamics
5. Analysis: Feature Quality
6. Comparison: Local vs Backprop
7. CIFAR-10 Attempt
8. Conclusions & Open Questions

## 1. Setup & Imports

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict

# Import our core implementation
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from core import EntropyGatedLayer, EntropyGatedNetwork, get_mnist, get_cifar10, DEVICE

# Also add benchmarks utils
sys.path.insert(0, os.path.join('..', '..', 'benchmarks'))

matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 11
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 2. Algorithm Explanation

### The Core Equation

Each neuron $i$ has a **plasticity gate** controlled by its local entropy:

$$\text{plasticity}_i = \sigma\Big(k \cdot (H_i - \theta)\Big)$$

where $H_i$ is the binary entropy of neuron $i$'s output across the batch:

$$H_i = -\frac{1}{B}\sum_{b=1}^{B} \Big[y_i^{(b)} \log y_i^{(b)} + (1 - y_i^{(b)}) \log (1 - y_i^{(b)})\Big]$$

### Three Local Learning Signals

| Signal | Equation | Role |
|--------|----------|------|
| Reconstruction | $\Delta W_{\text{recon}} = \frac{1}{B} y^T (x - yW)$ | Can I predict my input? |
| Decorrelation | $\Delta W_{\text{decorr}} = -(C_{\text{offdiag}} \cdot W)$ | Am I redundant? |
| Sparsity | $\Delta W_{\text{sparse}} = -(\bar{y} - \tau) \cdot W$ | Right firing rate? |

### Combined Update

$$\Delta W_i = \text{plasticity}_i \cdot \Big(0.50 \cdot \Delta W_{\text{recon}} + 0.25 \cdot \Delta W_{\text{decorr}} + 0.25 \cdot \Delta W_{\text{sparse}}\Big)$$

### Why It Works

**Implicit curriculum learning:**
- Easy patterns → neurons become confident (low entropy) → stop changing
- Hard patterns → neurons stay uncertain (high entropy) → keep learning
- No global loss function needed — each neuron self-regulates

## 3. Training Run (MNIST)

In [ ]:
# Load MNIST
train_loader, test_loader, input_dim, num_classes = get_mnist(256)
print(f'Train batches: {len(train_loader)}, Test batches: {len(test_loader)}')

In [ ]:
# Create model (V4 architecture: 700, 400)
model = EntropyGatedNetwork(
    input_dim=784, hidden_dims=[700, 400], num_classes=10,
    max_epochs=50, batches_per_epoch=len(train_loader)
).to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
probe_p = sum(p.numel() for p in model.probe.parameters())
print(f'Total params: {total_p:,}')
print(f'Local params: {total_p - probe_p:,} ({(total_p-probe_p)/total_p*100:.1f}%)')
print(f'Probe params: {probe_p:,} ({probe_p/total_p*100:.1f}%)')

In [ ]:
# Training loop with instrumentation
import time

history = {'epoch': [], 'loss': [], 'accuracy': [], 'time': [],
           'entropy_l1': [], 'entropy_l2': [],
           'plasticity_l1': [], 'plasticity_l2': [],
           'weight_norm_l1': [], 'weight_norm_l2': []}

N_EPOCHS = 50  # Use 100 for full reproduction of 97.46%

for ep in range(1, N_EPOCHS + 1):
    t0 = time.time()
    loss = model.train_epoch(train_loader)
    dt = time.time() - t0
    acc = model.evaluate(test_loader)
    
    # Record metrics
    history['epoch'].append(ep)
    history['loss'].append(loss)
    history['accuracy'].append(acc)
    history['time'].append(dt)
    
    # Instrument entropy and plasticity per layer
    with torch.no_grad():
        for i, layer in enumerate(model.layers):
            # Compute entropy on a test batch
            x_sample, _ = next(iter(test_loader))
            x_sample = x_sample.to(DEVICE).view(x_sample.size(0), -1)
            if i > 0:
                for j in range(i):
                    x_sample = model.layers[j](x_sample)
            y_sample = layer(x_sample)
            eps = 1e-7
            H = -(y_sample * torch.log(y_sample + eps) + (1-y_sample) * torch.log(1-y_sample + eps)).mean(0)
            plast = torch.sigmoid(5 * (H - 0.4))
            history[f'entropy_l{i+1}'].append(H.mean().item())
            history[f'plasticity_l{i+1}'].append(plast.mean().item())
            history[f'weight_norm_l{i+1}'].append(layer.W.data.norm().item())
    
    if ep <= 10 or ep % 5 == 0:
        print(f'Ep {ep:3d} | Loss {loss:.4f} | Acc {acc*100:.2f}% | {dt:.1f}s')

## 4. Analysis: Entropy & Plasticity Dynamics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Accuracy curve
axes[0, 0].plot(history['epoch'], [a*100 for a in history['accuracy']], 'b-', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Test Accuracy (%)')
axes[0, 0].set_title('Accuracy (no backprop in features)')
axes[0, 0].axhline(y=98.04, color='r', linestyle='--', alpha=0.7, label='Backprop ref')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss curve
axes[0, 1].plot(history['epoch'], history['loss'], 'g-', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Probe Loss')
axes[0, 1].set_title('Training Loss (probe only)')
axes[0, 1].grid(True, alpha=0.3)

# Time per epoch
axes[0, 2].plot(history['epoch'], history['time'], 'orange', linewidth=2)
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Time (s)')
axes[0, 2].set_title('Time per Epoch')
axes[0, 2].grid(True, alpha=0.3)

# Entropy per layer
axes[1, 0].plot(history['epoch'], history['entropy_l1'], 'b-', label='Layer 1', linewidth=2)
axes[1, 0].plot(history['epoch'], history['entropy_l2'], 'r-', label='Layer 2', linewidth=2)
axes[1, 0].axhline(y=0.4, color='k', linestyle='--', alpha=0.5, label='Threshold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Mean Entropy')
axes[1, 0].set_title('Per-Layer Entropy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plasticity per layer
axes[1, 1].plot(history['epoch'], history['plasticity_l1'], 'b-', label='Layer 1', linewidth=2)
axes[1, 1].plot(history['epoch'], history['plasticity_l2'], 'r-', label='Layer 2', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Mean Plasticity')
axes[1, 1].set_title('Per-Layer Plasticity (entropy gate)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Weight norms
axes[1, 2].plot(history['epoch'], history['weight_norm_l1'], 'b-', label='Layer 1', linewidth=2)
axes[1, 2].plot(history['epoch'], history['weight_norm_l2'], 'r-', label='Layer 2', linewidth=2)
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Weight Frobenius Norm')
axes[1, 2].set_title('Weight Stability')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Entropy-Gated Learning — Training Dynamics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/training_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Analysis: Feature Quality

Let's look at what the local learning layers actually learn.

In [ ]:
# Visualize Layer 1 filters (weight rows reshaped to 28x28)
W1 = model.layers[0].W.data.cpu().numpy()  # [700, 784]

fig, axes = plt.subplots(8, 16, figsize=(16, 8))
fig.suptitle('Layer 1 Filters (first 128 of 700) — learned WITHOUT backprop', fontsize=13)
for i in range(8):
    for j in range(16):
        idx = i * 16 + j
        w = W1[idx].reshape(28, 28)
        axes[i, j].imshow(w, cmap='RdBu_r', vmin=-np.percentile(np.abs(w), 95),
                          vmax=np.percentile(np.abs(w), 95))
        axes[i, j].axis('off')
plt.tight_layout()
plt.savefig('results/layer1_filters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-neuron entropy histogram
with torch.no_grad():
    x_all, y_all = [], []
    for x, y in test_loader:
        x_all.append(x); y_all.append(y)
    x_all = torch.cat(x_all).to(DEVICE).view(-1, 784)
    y_all = torch.cat(y_all).to(DEVICE)
    
    h1 = model.layers[0](x_all)
    h2 = model.layers[1](h1)
    
    eps = 1e-7
    H1 = -(h1 * torch.log(h1+eps) + (1-h1) * torch.log(1-h1+eps)).mean(0).cpu().numpy()
    H2 = -(h2 * torch.log(h2+eps) + (1-h2) * torch.log(1-h2+eps)).mean(0).cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(H1, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(x=0.4, color='red', linestyle='--', label='Threshold')
ax1.set_xlabel('Neuron Entropy')
ax1.set_ylabel('Count')
ax1.set_title(f'Layer 1 — Per-Neuron Entropy Distribution (n={len(H1)})')
ax1.legend()

ax2.hist(H2, bins=50, alpha=0.7, color='coral', edgecolor='black')
ax2.axvline(x=0.4, color='red', linestyle='--', label='Threshold')
ax2.set_xlabel('Neuron Entropy')
ax2.set_ylabel('Count')
ax2.set_title(f'Layer 2 — Per-Neuron Entropy Distribution (n={len(H2)})')
ax2.legend()

plt.suptitle('Entropy Distribution After Training — Low entropy = confident, stable neurons', fontsize=12)
plt.tight_layout()
plt.savefig('results/entropy_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Layer 1: {(H1 < 0.4).sum()} confident / {(H1 >= 0.4).sum()} plastic neurons')
print(f'Layer 2: {(H2 < 0.4).sum()} confident / {(H2 >= 0.4).sum()} plastic neurons')

In [ ]:
# Correlation matrix of Layer 2 activations (should be decorrelated)
with torch.no_grad():
    h2_np = h2.cpu().numpy()
    corr = np.corrcoef(h2_np.T)  # [400, 400]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

im1 = ax1.imshow(corr[:50, :50], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
ax1.set_title('Correlation Matrix (first 50 neurons)')
ax1.set_xlabel('Neuron')
ax1.set_ylabel('Neuron')
plt.colorbar(im1, ax=ax1)

# Histogram of off-diagonal correlations
mask = ~np.eye(corr.shape[0], dtype=bool)
offdiag = corr[mask].flatten()
ax2.hist(offdiag, bins=100, alpha=0.7, color='mediumpurple', edgecolor='black')
ax2.set_xlabel('Pairwise Correlation')
ax2.set_ylabel('Count')
ax2.set_title(f'Off-diagonal Correlations (mean={offdiag.mean():.4f}, std={offdiag.std():.4f})')
ax2.axvline(x=0, color='red', linestyle='--')

plt.suptitle('Layer 2 Decorrelation — Local anti-Hebbian keeps neurons independent', fontsize=12)
plt.tight_layout()
plt.savefig('results/decorrelation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparison: Local vs Backprop

In [ ]:
# Train a backprop baseline with the SAME architecture for fair comparison
class BackpropBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 700), nn.Sigmoid(),
            nn.Linear(700, 400), nn.Sigmoid(),
            nn.Linear(400, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 10)
        )
        self.opt = torch.optim.Adam(self.parameters(), lr=1e-3, weight_decay=1e-4)
    
    def forward(self, x): return self.net(x)
    
    def train_epoch(self, loader):
        self.train(); tl = 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            self.opt.zero_grad()
            loss = F.cross_entropy(self(x), y)
            loss.backward(); self.opt.step(); tl += loss.item()
        return tl / len(loader)
    
    def evaluate(self, loader):
        self.eval(); c = t = 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                c += (self(x).argmax(1) == y).sum().item(); t += y.size(0)
        return c / t

bp_model = BackpropBaseline().to(DEVICE)
bp_history = []

for ep in range(1, N_EPOCHS + 1):
    loss = bp_model.train_epoch(train_loader)
    acc = bp_model.evaluate(test_loader)
    bp_history.append(acc)
    if ep <= 5 or ep % 10 == 0:
        print(f'Backprop Ep {ep:3d} | Loss {loss:.4f} | Acc {acc*100:.2f}%')

In [ ]:
# Side-by-side comparison
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(history['epoch'], [a*100 for a in history['accuracy']],
        'b-', linewidth=2.5, label='Entropy-Gated (LOCAL)', marker='o', markersize=3)
ax.plot(range(1, len(bp_history)+1), [a*100 for a in bp_history],
        'r--', linewidth=2.5, label='Backprop (GLOBAL)', marker='s', markersize=3)

ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Test Accuracy (%)', fontsize=13)
ax.set_title('Local Learning vs Backpropagation — Same Architecture [700, 400]', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Annotate final scores
eg_final = history['accuracy'][-1] * 100
bp_final = bp_history[-1] * 100
ax.annotate(f'EG: {eg_final:.2f}%', xy=(N_EPOCHS, eg_final),
            xytext=(N_EPOCHS-10, eg_final-3), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='blue'), color='blue')
ax.annotate(f'BP: {bp_final:.2f}%', xy=(N_EPOCHS, bp_final),
            xytext=(N_EPOCHS-10, bp_final+2), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='red'), color='red')

plt.tight_layout()
plt.savefig('results/local_vs_backprop.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nEntropy-Gated (local): {eg_final:.2f}%')
print(f'Backprop (global):     {bp_final:.2f}%')
print(f'Gap:                   {bp_final - eg_final:.2f}%')

## 7. CIFAR-10 Attempt

CIFAR-10 is the open challenge. The MLP version reaches ~41% (flatten),  
the Conv V2 (ReLU+LocalBN) reaches 48.22%.  
Backprop reference: 85.83%.

**The gap is due to:**
1. Simplified channel-wise Hebbian loses spatial correlations
2. No spatial inductive bias in the local learning rule
3. Deeper conv stacks suffer from credit assignment without backprop

In [ ]:
# Quick CIFAR-10 test with MLP (flatten)
train_c, test_c, input_dim_c, nc_c = get_cifar10(128)

model_cifar = EntropyGatedNetwork(
    input_dim=3072, hidden_dims=[1000, 500], num_classes=10,
    max_epochs=20, batches_per_epoch=len(train_c)
).to(DEVICE)

print('CIFAR-10 MLP (20 epochs):')
for ep in range(1, 21):
    loss = model_cifar.train_epoch(train_c)
    acc = model_cifar.evaluate(test_c)
    if ep <= 5 or ep % 5 == 0:
        print(f'  Ep {ep:2d} | Loss {loss:.4f} | Acc {acc*100:.2f}%')

## 8. Conclusions & Open Questions

### Results Summary

| Dataset | EG (local) | Backprop (global) | Gap |
|---------|-----------|-------------------|-----|
| **MNIST** | **97.46%** | 98.04% | **0.58%** |
| CIFAR-10 (MLP) | 41.16% | — | — |
| CIFAR-10 (Conv) | 48.22% | 85.83% | 37.6% |

### Key Insight
Entropy-gated plasticity creates an **implicit curriculum**: neurons self-regulate their learning rate based on uncertainty, no global loss needed.

### Open Questions
1. **CIFAR-10 gap**: Need better spatial Hebbian for conv layers
2. **Depth**: Can EG scale beyond 2 layers?
3. **Fully local**: Can we replace the probe MLP with a local classifier?
4. **Theory**: What is the theoretical connection to free energy minimization?
5. **Speed**: Can local learning be parallelized better than backprop?